# Constant Optimization

After the evolutionary search finds a good DAG topology, the scalar
constants inside `ExpAffine` and `ConstantBrick` nodes can be refined
via dichotomy line search. This notebook shows the rational-only
constant-optimization workflow.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, os
from pathlib import Path

notebook_dir = Path().resolve()
repo_root = notebook_dir.parents[2] if notebook_dir.name == "SymbolicRegression" else notebook_dir
lib_dir = repo_root / "lib"
if str(lib_dir) not in sys.path:
    sys.path.insert(0, str(lib_dir))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## 1. Generate Data

In [ ]:
R  = 8.314
n_samples = 2000
rng = np.random.default_rng(42)

P  = rng.uniform(1e4, 1e6, n_samples)
nc = rng.uniform(0.1, 10,  n_samples)
T  = rng.uniform(200, 1000, n_samples)

X = pd.DataFrame({"P": P, "n": nc, "T": T})
y = pd.Series(nc * R * T / P, name="V")

## 2. Feature Selection + Search Space

In [ ]:
from dragon.utils.symbolic.features_selection import GradientBoostingSelector, CombinationBuilder
from dragon.search_space.bricks import (
    Identity, SelectFeatures, Inverse, Negate, Power,
    SumFeatures, ConstantBrick, Ln, Sin, Cos, Exp, ExpAffine, ChannelBoost,
)
from dragon.search_space.bricks_variables import operations_var
from dragon.search_space.base_variables import CatVar, Constant, ArrayVar
from dragon.search_space.dag_encoding import SymbolicNode, AdjMatrix
from dragon.search_space.dag_variables import HpVar, EvoDagVariable
from dragon.search_operators.base_neighborhoods import (
    CatInterval, ConstantInterval, ArrayInterval,
)
from dragon.search_operators.dag_neighborhoods import EvoDagInterval, HpInterval

selector = GradientBoostingSelector(n_top=3, model_type="xgboost")
top_features, scores = selector.fit_transform(X, y)
num_features = len(top_features)

combo_builder = CombinationBuilder()
all_combos = combo_builder.build(top_features, scores)
combo_weights = SelectFeatures.combination_weights(
    [scores.get(c, 1.0 / num_features) for c in top_features], all_combos)

def hpv(label, brick, hps=None):
    return HpVar(label, brick, hyperparameters=hps or {}, neighbor=HpInterval())
def const(label, cls):
    return Constant(label, cls, neighbor=ConstantInterval())
def cat(label, features, weights=None):
    return CatVar(label, features=features, weights=weights, neighbor=CatInterval())

MAX_NODES = 15
cand_ops = [
    hpv("SelectFeatures", const("SelectFeaturesOp", SelectFeatures),
        {"feature_indices": cat("feature_indices", all_combos, weights=combo_weights)}),
    hpv("UnaryOp",  cat("UnaryOpType", [Identity, Inverse, Negate])),
    hpv("Power",    const("PowerOp", Power),
        {"exponent": cat("exponent", [-3, -2, -1, 1, 2, 3])}),
    hpv("Sum",      const("SumOp", SumFeatures)),
    hpv("Ln",       const("LnOp", Ln)),
    hpv("Sin",      const("SinOp", Sin)),
    hpv("Cos",      const("CosOp", Cos)),
    hpv("Exp",      const("ExpOp", Exp)),
    hpv("ExpAffine",const("ExpAffineOp", ExpAffine)),
    hpv("ChannelBoost", const("ChannelBoostOp", ChannelBoost),
        {"mode": cat("mode", ["add", "sub", "mul", "div"])}),
    hpv("ConstantBrick", const("ConstOp", ConstantBrick)),
]
operations = operations_var(
    "CandidateOperations", size=MAX_NODES, candidates=cand_ops,
    combiner_features=["add", "mul"],
    activations=Constant("id", value=nn.Identity(), neighbor=ConstantInterval()),
    node_type=SymbolicNode,
)
dag = EvoDagVariable(label="Dag", operations=operations, init_complexity=4,
                     neighbor=EvoDagInterval(nb_mutations=2))
search_space = ArrayVar(dag, label="Search Space", neighbor=ArrayInterval())

## 3. Dataset

In [ ]:
class MetaArchi(nn.Module):
    def __init__(self, args, input_shape):
        super().__init__()
        self.dag = args['Dag']
        self.dag.set(input_shape)
    def forward(self, X):
        return self.dag(X)


class RegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values)
        self.y = torch.FloatTensor(y.values)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(RegressionDataset(X, y), batch_size=256, shuffle=False)


def forward(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            preds.append(model(Xb))
            trues.append(yb.reshape(-1, 1))
    return torch.cat(preds), torch.cat(trues)

## 4. Build a Probe Batch + probe_fn

The constant optimizer needs a fixed batch to evaluate against.
`probe_fn` is user-written: it scores a model on the fixed batch
so `sweep_constants` can rank candidate constant values.

In [ ]:
def make_const_batch(X_np, y_np, device, max_points=512):
    """Sample a fixed batch for constant optimization."""
    n = len(y_np)
    m = min(n, max_points)
    rng = np.random.default_rng(0)
    sub = rng.choice(n, m, replace=False) if n > m else np.arange(n)
    Xb = torch.tensor(X_np[sub], dtype=torch.float32, device=device)
    yb = torch.tensor(y_np[sub], dtype=torch.float32, device=device).reshape(-1, 1)
    return Xb, yb


X_np = X.values.astype(np.float64)
y_np = y.values.astype(np.float64)
device = torch.device("cpu")

const_batch = make_const_batch(X_np, y_np, device)
print(f"Constant-opt batch: X={const_batch[0].shape}, y={const_batch[1].shape}")

In [ ]:
def make_probe_fn(Xb, yb):
    """Return a probe function that scores the model on a fixed batch.
    
    Uses correlation-based scoring so sweep_constants can rank candidates.
    """
    def probe(model):
        model.eval()
        with torch.no_grad():
            pred = model(Xb)
        if pred.ndim == 1:
            pred = pred.reshape(-1, 1)
        P = pred.detach().cpu().numpy()
        y = yb.detach().cpu().numpy().ravel()
        n_ch = P.shape[1]
        total = 0.0
        for c in range(n_ch):
            raw = P[:, c]
            if not np.all(np.isfinite(raw)) or np.std(raw) < 1e-12:
                total += 1.0
                continue
            A = np.column_stack([raw, np.ones_like(raw)])
            try:
                coef, *_ = np.linalg.lstsq(A, y, rcond=None)
                yp = coef[0] * raw + coef[1]
            except Exception:
                yp = raw
            if np.std(yp) < 1e-12:
                total += 1.0
            else:
                total += 1.0 - abs(np.corrcoef(yp, y)[0, 1])
        return total / max(n_ch, 1)
    return probe


probe_fn = make_probe_fn(*const_batch)

## 5. Loss Function with Constant Optimization

After the forward pass, `sweep_constants` refines all `ExpAffine.a`
and `ConstantBrick.value` parameters via golden-section line search.
The loss is then computed from the OLS-combined prediction.

In [ ]:
from dragon.utils.symbolic.loss_function.ols_pipeline import evaluate as ols_evaluate
from dragon.search_space.bricks.symbolic_regression import sweep_constants
from dragon.utils.symbolic.dag_to_formula import graph_to_all_formulas
from dragon.utils.symbolic.formula_extraction import (
    channel_formula, ols_formula, set_best_formula,
)


def loss_function(args, idx):
    labels = [e.label for e in search_space]
    args_dict = ({labels[0]: args} if isinstance(args, AdjMatrix)
                 else dict(zip(labels, args)))

    model = MetaArchi(args_dict, input_shape=(num_features,))

    sweep_constants(model, probe_fn, iters=40, max_mag=1e12, sweeps=1)

    pred_all, true_all = forward(model, train_loader)
    loss, selected_c, ols_w, _, _, _, _, valid_idx, analysis = \
        ols_evaluate(pred_all, true_all, search_loss='corr', loss_mode='full')

    formulas = graph_to_all_formulas(
        model.dag.matrix, top_features, model.dag.operations, parse_sympy=False)

    f = ols_formula(formulas, analysis)
    if f is None:
        f = channel_formula(formulas, selected_c, pred_all,
                            true_all.squeeze().numpy(), 'corr') or 'N/A'

    if idx % 5 == 0:
        print(f"  [{idx}] loss={loss:.8f}  formula={str(f)[:80]}")

    return loss, model


## 6. Run the Search

In [ ]:
from dragon.search_algorithm.ssea import SteadyStateEA

algo = SteadyStateEA(
    search_space, n_iterations=30, population_size=10, selection_size=5,
    evaluation=loss_function, save_dir="save/constopt",
)
min_loss = algo.run()
print(f"\nBest loss: {min_loss}")

## 7. Inspect Results

In [ ]:
from dragon.utils.plot_functions import load_archi
from dragon.utils.symbolic.dag_to_formula import graph_to_all_formulas
from dragon.utils.symbolic.formula_extraction import ols_formula, channel_formula

best = load_archi("save/constopt/best_model/x.pkl")
if isinstance(best, list):
    best = best[0]
print(best)

pred, true = forward(best, train_loader)
_, selected_c, ols_w, _, _, _, _, valid_idx, analysis = \
    ols_evaluate(pred, true, search_loss='corr', loss_mode='full')
formulas = graph_to_all_formulas(
    best.matrix, top_features, best.operations, parse_sympy=False)
f = ols_formula(formulas, analysis)
if f is None:
    f = channel_formula(formulas, selected_c, pred,
                        true.squeeze().numpy(), 'corr') or 'N/A'
print(f"\nExtracted formula: {f}")
print(f"True formula:     V = n * R * T / P")
